In [ ]:
import os 
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
NVIDIA_BASE_URL= "https://integrate.api.nvidia.com/v1"
load_dotenv(override=True)
meta_api_key = os.getenv("META_API_KEY")
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
meta = OpenAI(base_url = NVIDIA_BASE_URL, api_key = meta_api_key)

In [ ]:
system_message = "You are a helpful assistant"
gemini_model="gemini-2.5-flash-lite"
def message_gemini(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
system_message = "You are a helpful assistant"

def message_meta(prompt):
    messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
    response = meta.chat.completions.create(model="meta/llama-3.3-70b-instruct", messages=messages)
    return response.choices[0].message.content

In [ ]:
message_gemini("What's today's date?")

In [ ]:
def shout(text):
    print(f"Shout has been called with input {text}")
    return text.upper()

In [ ]:
shout("hello")

In [ ]:
gr.Interface(fn=shout, inputs="textbox", outputs="textbox", flagging_mode="never").launch()

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=shout,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be shouted", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=message_gemini,
    title="Shout", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["hello", "howdy"], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"
message_input = gr.Textbox(label="Your message:", info="Enter a message for gemini-2.5-flash-lite", lines=7)
message_output = gr.Markdown(label="Response:")


view = gr.Interface(
    fn=message_gemini,
    title="GEMINI", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"
message_input = gr.Textbox(label="Your message:", info="Enter a message for gemini-2.5-pro", lines=7)
message_output = gr.Markdown(label="Response:")

gemini_model="gemini-2.5-pro"
view = gr.Interface(
    fn=message_gemini,
    title="GEMINI", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
def stream_gemini(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = gemini.chat.completions.create(
        model=gemini_model,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"
message_input = gr.Textbox(label="Your message:", info="Enter a message for gemini-2.5-pro", lines=7)
message_output = gr.Markdown(label="Response:")

gemini_model="gemini-3.1-pro-preview"
view = gr.Interface(
    fn=stream_gemini,
    title="GEMINI", 
    inputs=[message_input], 
    outputs=[message_output], 
    examples=[
        "Explain the Transformer architecture to a layperson",
        "Explain the Transformer architecture to an aspiring AI engineer",
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
def stream_meta(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
      ]
    stream = meta.chat.completions.create(
        model="meta/llama-3.3-70b-instruct",
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result
    

In [ ]:
def stream_model(prompt,model):
    if model == "GEMINI":
        result = stream_gemini(prompt)
    elif model == "META":
        result = stream_meta(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for the LLM", lines=7)
model_selector = gr.Dropdown(["GEMINI", "META"], label="Select model", value="GEMINI")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Explain the Transformer architecture to a layperson", "GEMINI"],
            ["Explain the Transformer architecture to an aspiring AI engineer", "META"]
        ], 
    flagging_mode="never"
    )
view.launch()

In [ ]:
from web_scraper import fetch_website_contents

In [ ]:

system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def stream_brochure(company_name, url, model):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    if model=="GEMINI":
        result = stream_gemini(prompt)
    elif model=="META":
        result = stream_meta(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result

In [ ]:
gemini_model="gemini-2.5-flash-lite"
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
model_selector = gr.Dropdown(["GEMINI", "META"], label="Select model", value="GEMINI")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Hugging Face", "https://huggingface.co", "GEMINI"],
            ["Edward Donner", "https://edwarddonner.com", "META"]
        ], 
    flagging_mode="never"
    )
view.launch()